In [1]:
import requests
from dotenv import load_dotenv
import os
import json
import time
from typing import Dict

load_dotenv()

api_key = os.getenv("RAPID_API_KEY")

In [2]:
with open("manhattan_listings.json", "r", encoding="utf-8") as f:
    listings = json.load(f)

ids = [listing["id"] for listing in listings if "id" in listing]

print(f"Extracted {len(ids)} listing IDs")

# check if ids are unique
if len(ids) == len(set(ids)):
    print("Ids are all unique")
else:
    print("IDS ARE NOT UNIQUE. PLEASE CORRECT BEFORE GETTING APARTMENT DETAILS")

Extracted 6170 listing IDs
Ids are all unique


In [3]:
def load_existing_details() -> Dict:
    """Load existing apartment details."""
    if os.path.exists("manhattan_details.json"):
        with open("manhattan_details.json", "r", encoding="utf-8") as f:
            details = json.load(f)
            return {str(item["id"]): item for item in details}
    return {}

# Replace the second cell with:
# Load changed listings
with open("changed_listings.json", "r", encoding="utf-8") as f:
    changed_listings = json.load(f)

# Load current listings to know what should be deleted
with open("manhattan_listings.json", "r", encoding="utf-8") as f:
    current_listings = json.load(f)
    current_ids = set(str(listing["id"]) for listing in current_listings)

# Load existing details
existing_details = load_existing_details()

# Identify listings to process
new_ids = [str(listing["id"]) for listing in changed_listings]
print(f"Found {len(new_ids)} listings to update")

# Identify listings to remove
removed_ids = set(existing_details.keys()) - current_ids
if removed_ids:
    print(f"Found {len(removed_ids)} listings to remove")

Found 3718 listings to update
Found 3972 listings to remove


In [4]:
headers = {
    "x-rapidapi-key": api_key,
    "x-rapidapi-host": "streeteasy-api.p.rapidapi.com"
}

# Fetch details for new/updated listings
for i, listing_id in enumerate(new_ids, 1):
    url = f"https://streeteasy-api.p.rapidapi.com/rentals/{listing_id}"
    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()
        existing_details[listing_id] = r.json()
        print(f"Fetched {i}/{len(new_ids)}: {listing_id}")
    except requests.RequestException as e:
        print(f"Error fetching {listing_id}: {e}")
        continue
    time.sleep(0.2)

# Remove listings that no longer exist
for removed_id in removed_ids:
    existing_details.pop(removed_id, None)

# Save updated details
with open("manhattan_details.json", "w", encoding="utf-8") as f:
    json.dump(list(existing_details.values()), f, indent=2)

print(f"Updated details: {len(new_ids)} new/changed, {len(removed_ids)} removed")

Fetched 1/3718: 4915438
Fetched 2/3718: 4915425
Fetched 3/3718: 4915423
Fetched 4/3718: 4915413
Fetched 5/3718: 4915412
Fetched 6/3718: 4915395
Fetched 7/3718: 4915385
Fetched 8/3718: 4915384
Fetched 9/3718: 4915379
Fetched 10/3718: 4915378
Fetched 11/3718: 4915376
Fetched 12/3718: 4915375
Fetched 13/3718: 4915374
Fetched 14/3718: 4915373
Fetched 15/3718: 4915372
Fetched 16/3718: 4915370
Fetched 17/3718: 4915367
Fetched 18/3718: 4915364
Fetched 19/3718: 4915362
Fetched 20/3718: 4915360
Fetched 21/3718: 4915359
Fetched 22/3718: 4915335
Fetched 23/3718: 4915334
Fetched 24/3718: 4915329
Fetched 25/3718: 4915328
Fetched 26/3718: 4915327
Fetched 27/3718: 4915326
Fetched 28/3718: 4915317
Fetched 29/3718: 4915315
Fetched 30/3718: 4915312
Fetched 31/3718: 4915310
Fetched 32/3718: 4915305
Fetched 33/3718: 4915304
Fetched 34/3718: 4915302
Fetched 35/3718: 4915301
Fetched 36/3718: 4915299
Fetched 37/3718: 4915294
Fetched 38/3718: 4915284
Fetched 39/3718: 4915278
Fetched 40/3718: 4915274
Fetched 4

In [5]:
# converts json to csv, no longer needed


# with open("manhattan_details.json", "r") as f:
#     data = json.load(f)

# df = pd.json_normalize(
#     data,
#     sep="_",  # replaces nested keys with underscore, e.g. building_id
# )

# # Convert list-type columns to comma-separated strings
# for col in df.columns:
#     df[col] = df[col].apply(
#         lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
#     )

# # Save to CSV
# df.to_csv("manhattan_details.csv", index=False)
# print("json to csv conversion successful")